# Data Coverage

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without resetting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

Time Differences

In [ ]:
%store -r restaurant_data_unprocessed
timezones_acronyms = {}
for loc_id, df in restaurant_data_unprocessed.items():
    time = df['created_at'].iloc[0]
    timezone = time.strip('0123456789-+: ')
    timezones_acronyms[loc_id] = timezone
timezones = {
    '0RJH3FFPYBPEY': 'America/New_York',
    '1SQPTEGYPH0GA': 'America/Denver',
    '3AXDVZJYN9DRS': 'Europe/London',
    '75WYSXR9QBK5M': 'Pacific/Honolulu',
    '78AY09MVJVTYE': 'America/New_York',
    '9XKJD8DQTH559': 'America/New_York',
    'AQD04SM0J92WA': 'America/Los_Angeles',
    'CB2KHY1C2G9PT': 'America/New_York',
    'EMBVNVD207CC6': 'America/New_York',
    'JHDN7CF1C03X5': 'America/Chicago',
    'L3XS7WSJ4AJA3': 'Europe/London',
    'L69HYJ4Y3TR91': 'America/New_York',
    'LBMCPAYT7W36V': 'America/New_York',
    'LBZEEFSBJNB3Z': 'America/Los_Angeles',
    'LFZFT3VASXPED': 'Australia/Sydney',
    'LQ5EH4BKGV61T': 'America/New_York',
    'LZ5MR1TS37E7W': 'America/Los_Angeles',
    'MS8R16DY0JQAM': 'America/Los_Angeles',
    'N0PC58FB2XAZ3': 'America/Chicago',
    'S8MT0YGD2KTN9': 'America/New_York',
    'SAFK7ND1HR6XS': 'America/Los_Angeles',
    'SRQS8F7JWA9MZ': 'America/New_York',
    'V3Q26BHF3SE2H': 'America/New_York',
    'W8T41JZK0ZMEP': 'America/New_York',
    'WJA3YCD4QBWRX': 'America/New_York',
    '1G5AJ17XCH2A8': 'America/Chicago',
    'ADPFRN3QZRCXK': 'America/Los_Angeles',
    'ED5J990H5VAZT': 'America/Los_Angeles',
    '2HRX9P6HKXA8V': 'America/Los_Angeles',
    'C0BE4NDSW26QN': 'America/New_York'
}
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])

### Calculating Weekly Data Coverage

In [ ]:
# Calculate active weeks
active_weeks_dict = {}
for loc_id, df in sales_and_menu_data.items():
    active_weeks_dict[loc_id] = (df
                                 .resample('W')
                                 .size()
                                 .to_frame(name='W')
                                 .query('0 < W')
                                 .index
                                 .tz_localize(None)
                                 .to_period('W')
                                 .tolist())

### Weekly Data Coverage Visual

In [ ]:
# Visualizing with gaps for inactive weeks
coverage_fig, ax = plt.subplots(figsize=(14, 8))

# Loop through every active week within a single restaurant
for loc_id, active_weeks in active_weeks_dict.items():

    # Index into the promotional items for this restaurant
    promo_datetime = before_after_details.loc[loc_id,'cross_over_date'].tz_localize('UTC')

    # For every active week
    for week in active_weeks:

        # Place a blue dot
        ax.hlines(y=loc_id, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=loc_id)

    # Place a red circle for the promotional item
    ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title('Weekly Activity for Each Restaurant with Gaps for Inactive Weeks')
ax.set_xlabel('Date')
ax.set_ylabel('Restaurant ID')

# Figure
coverage_fig.tight_layout()

# Save
plt.savefig('Data Coverage.png', bbox_inches='tight')
plt.show()

### Data Density: Entire Set, Before Promo, and After Promo

In [ ]:
def coverage_calculator(loc_id, df, freqs=['W-MON','D','12H','6H'], periods=['all','4mo','bef','b2m','aft','a2m']):

    # Remove duplicates orders to count distinct number of orders
    df = df.drop_duplicates('order_id')

    # Identify necessary dates
    first_date = df.index[0]
    last_date = df.index[-1]
    promo_datetime = before_after_details.loc[loc_id, 'cross_over_date'].tz_localize(timezones[loc_id]) # Identify introduction date
    two_months_before = promo_datetime - pd.DateOffset(days=60) # Two months before the promotional introduction date
    two_months_after = promo_datetime + pd.DateOffset(days=60) # Two months after the promotional introduction date

    # All time period options
    all_periods = {'all':(first_date, last_date),
                   '4mo':(two_months_before, two_months_after),
                   'bef':(first_date, promo_datetime),
                   'b2m':(two_months_before, promo_datetime),
                   'aft':(promo_datetime, last_date),
                   'a2m':(promo_datetime, two_months_after)}

    # Filter to chosen time periods for iterating
    periods_to_use = {}
    for period in periods:
        periods_to_use[period] = all_periods[period]

    # Initialize container to aggregate for summary
    row = {'loc_id': loc_id}
    for qualifier, (beginning, end) in periods_to_use.items():
        
        # Unpack beginning and end dates to determine the actual data within the period
        period = df.loc[beginning:end]
        
        # Calculate total
        row[f'{qualifier}_order'] = period.shape[0]

        # Resample the time series for every frequency
        for freq in freqs:

            # Number of active weeks within the bounds, and then before and after
            resampled_data = period.resample(freq).size()
            active_total_at_freq = (0 < resampled_data).sum()

            if 'W' in freq:
                end = end + pd.DateOffset(weeks=1)

            # Possible periods
            possible = pd.date_range(beginning, end, freq=freq, ambiguous=True, inclusive='left')
            possible_total = possible.shape[0]
            if 'H' in freq and beginning.utcoffset() > end.utcoffset():
                possible_total -= 1
            
            # Calculate data coverage (when the restaurant is active) as a fraction of the total possible days
            coverage_ratio = active_total_at_freq / possible_total
            rounded_coverage_ratio = float(int(round(100*coverage_ratio)))/100

            # Other simple stats
            rounded_mean = round(np.mean(resampled_data))
            rounded_sd = round(np.std(resampled_data))

            # Store
            # row[f'{qualifier}_{freq}'] = active_total_at_freq
            row[f'{qualifier}_{freq[:3]}_cover'] = rounded_coverage_ratio
            # row[f'{qualifier}_{freq}_mean'] = rounded_mean 
            # row[f'{qualifier}_{freq}_sd'] = rounded_sd

    return row

Apply function

In [ ]:
# Initialize a list to see if there is a data buffer before and after the promotional item introduction
data_coverage_list_4m = []
data_coverage_list_b2m = []
data_coverage_list_a2m = []
data_coverage_list_all = []
data_coverage_list_before = []
data_coverage_list_after = []

for loc_id, df in sales_and_menu_data.items():
    row_4m = coverage_calculator(loc_id, df, periods=['4mo'])
    row_b2m = coverage_calculator(loc_id, df, periods=['b2m'])
    row_a2m = coverage_calculator(loc_id, df, periods=['a2m'])
    row_all = coverage_calculator(loc_id, df, periods=['all'])
    row_before = coverage_calculator(loc_id, df, periods=['bef'])
    row_after = coverage_calculator(loc_id, df, periods=['aft'])
    
    data_coverage_list_4m.append(row_4m)
    data_coverage_list_b2m.append(row_b2m)
    data_coverage_list_a2m.append(row_a2m)
    data_coverage_list_all.append(row_all)
    data_coverage_list_before.append(row_before)
    data_coverage_list_after.append(row_after)

# Create data frame
data_coverage_4m = pd.DataFrame(data_coverage_list_4m).sort_values('4mo_12H_cover', ascending=False).set_index('loc_id')
restaurants_by_4m_coverage = data_coverage_4m.index.tolist()
data_coverage_b2m = pd.DataFrame(data_coverage_list_b2m).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_a2m = pd.DataFrame(data_coverage_list_a2m).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_all = pd.DataFrame(data_coverage_list_all).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_before = pd.DataFrame(data_coverage_list_before).set_index('loc_id').loc[restaurants_by_4m_coverage]
data_coverage_after = pd.DataFrame(data_coverage_list_after).set_index('loc_id').loc[restaurants_by_4m_coverage]

In [ ]:
before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'] = before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'].str.title()

In [ ]:
# sales_and_menu_data['SRQS8F7JWA9MZ'].resample('12H')['item_quantity'].sum().loc[before_after_details.loc['SRQS8F7JWA9MZ','cross_over_date'].tz_localize(timezones['SRQS8F7JWA9MZ'])-pd.DateOffset(days=60):before_after_details.loc['SRQS8F7JWA9MZ','cross_over_date'].tz_localize(timezones['SRQS8F7JWA9MZ'])+pd.DateOffset(days=60)]
# sales_and_menu_data['SRQS8F7JWA9MZ'].loc[before_after_details.loc['SRQS8F7JWA9MZ','cross_over_date'].tz_localize(timezones['SRQS8F7JWA9MZ'])-pd.DateOffset(days=60):before_after_details.loc['SRQS8F7JWA9MZ','cross_over_date'].tz_localize(timezones['SRQS8F7JWA9MZ'])+pd.DateOffset(days=60)].groupby('order_id', observed=True)['item_name'].count()

In [ ]:
possible = pd.date_range(before_after_details.loc['0RJH3FFPYBPEY','cross_over_date'].tz_localize(timezones['0RJH3FFPYBPEY'])-pd.DateOffset(days=60), 
                         before_after_details.loc['0RJH3FFPYBPEY','cross_over_date'].tz_localize(timezones['0RJH3FFPYBPEY'])+pd.DateOffset(days=60), 
                         freq='12H', ambiguous=True, inclusive='left')

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)
data_coverage_all.loc[restaurants_by_4m_coverage]

Visuals

In [ ]:
def plot_time_series(loc_id, df, max_ylim=0, freq='D', subset=True):

    
    # Turn auto display
    plt.ioff()

    # Filter to plant-based items and resample
    plant_based = df.query('is_plant_based == "Yes"')
    promo_datetime = pd.to_datetime(before_after_details.loc[loc_id, 'cross_over_date']).tz_localize('UTC')
    two_months_before = promo_datetime - pd.DateOffset(months=2)
    two_months_after = promo_datetime + pd.DateOffset(months=2)

    # Resample to specified frequency
    plant_based = plant_based.resample(freq)[['item_quantity']].sum()
    all_items = df.resample(freq)[['item_quantity']].sum()

    # Compute ratios and handle missing data
    all_items.replace(0, np.nan, inplace=True)
    plant_based_ratio = plant_based / all_items

    # Subset for the specified time range
    if subset:
        plant_based_ratio = plant_based_ratio[two_months_before:two_months_after]
        all_items = all_items[two_months_before:two_months_after]

    # Identify ends of contiguous data chunks
    is_contiguous = plant_based_ratio.notna()
    shift_plus = is_contiguous.shift(1, fill_value=False)
    shift_minus = is_contiguous.shift(-1, fill_value=False)
    start_points = is_contiguous & ~shift_plus
    end_points = is_contiguous & ~shift_minus

    # Plotting
    fig, ax = plt.subplots(1, 2, figsize=(16, 4))  # Creates a single subplot
    ax1, ax2 = ax

    ## Plot 1

    # Main plot
    ax1.plot(plant_based_ratio.index, plant_based_ratio, marker='o', markersize=1, linewidth=2, label='Plant-Based Items Fraction')

    # Adding dots for the start and end of each contiguous chunk
    ax1.plot(plant_based_ratio[start_points].index, plant_based_ratio[start_points], linewidth=0, color='#2fb7bf', markersize=5, marker='o', label='Start Extant Data')
    ax1.plot(plant_based_ratio[end_points].index, plant_based_ratio[end_points], linewidth=0, color='#1f77c4', markersize=5, marker='o', label='End Extant Data')

    # Promo date line
    ax1.axvline(x=promo_datetime, color='red', linestyle='--', label='Promo Date')

    # Ticks and limits
    xticks = pd.date_range(promo_datetime - pd.DateOffset(days=60), periods=10, freq="15D")
    if subset:
        ax1.set_xticks(ticks=xticks)
        ax1.set_xticklabels(labels=xticks.date, rotation=70)
        ax1.set_xlim(promo_datetime - pd.DateOffset(days=60), promo_datetime + pd.DateOffset(days=60))
    ax1.set_ylim(0, 1)

    # Axis and title
    ax1.set_title(f'Plant-Based Items Fraction for {loc_id}')
    ax1.set_ylabel('Ratio')
    ax1.set_xlabel('Date')
    ax1.legend()

    ## Plot 2

    # Main plot
    ax2.plot(all_items.index, all_items, linewidth=2, color='orange', marker='o', markersize=1, label='Total Item Quantity')

    # Extra details, adding dots to the edge of non-missing data
    ax2.axvline(x=promo_datetime, color='red', linestyle='--', label='Promo Date')

    # Adding dots for the start and end of each contiguous chunk
    ax2.plot(all_items[start_points].index, all_items[start_points], linewidth=0, color='#ff8500', markersize=5, marker='o', label='Start Extant Data')
    ax2.plot(all_items[end_points].index, all_items[end_points], linewidth=0, color='#ffa500', markersize=5, marker='o', label='End Extant Data')

    # Ticks and limits
    xticks = pd.date_range(promo_datetime - pd.DateOffset(days=60), periods=10, freq="15D")
    if subset:
        ax2.set_xticks(ticks=xticks)
        ax2.set_xticklabels(labels=xticks.date, rotation=70)
        ax2.set_xlim(promo_datetime - pd.DateOffset(days=60), promo_datetime + pd.DateOffset(days=60))
    ax2.set_ylim(0, max(max_ylim, all_items['item_quantity'].max()))

    # Axis and title
    ax2.set_title(f'Total Item Quantity Sold for {loc_id}')
    ax2.set_ylabel('Quantity')
    ax2.set_xlabel('Date')
    ax2.legend()

    # Save file externally
    #plt.savefig(f"Restaurant Sales Near Promo Introduction {loc_id}.png", bbox_inches='tight')

    return fig

In [ ]:
%store -r time_differences
%store -r time_differences_details

In [ ]:
if 'time_differences_details' not in locals() or 'time_differences' not in locals():

    time_differences_details = {}
    time_differences = {}
    for loc_id in restaurants_by_4m_coverage:

        df = sales_and_menu_data[loc_id]

        # Group by transactions (at the same time)
        transactions = df.groupby('created_at').agg({'item_quantity' : 'sum'})

        # Group by individuals days and days of the week
        transaction_by_dayofweek = transactions.groupby([transactions.index.dayofweek, transactions.index.date])

        # Take the index at every group and find the difference between time points (dropping the NaT edges) and convert to hours
        time_diffs_on_dayofweek = transaction_by_dayofweek.apply(lambda s: s.index.to_series().diff().dropna().dt.seconds//3600)

        existing_combinations = time_diffs_on_dayofweek.index.drop_duplicates()

        # Create a new MultiIndex from all days and existing (date, datetime) combinations
        all_days = np.arange(7) 
        new_indices = [(day, date, datetime) for day in all_days for _, date, datetime in existing_combinations]
        time_diffs_on_dayofweek = time_diffs_on_dayofweek.reindex(new_indices)

        # Store entire pivoted data
        time_differences_details[loc_id] = time_diffs_on_dayofweek

        time_diff_frequencies_list = []
        for dayofweek in range(7):

            # Subset to given day of the week and calculate the frequencies
            if dayofweek in time_diffs_on_dayofweek.index.get_level_values(0):
                time_diff_frequencies_specific_day = time_diffs_on_dayofweek[dayofweek].value_counts()
                time_diff_frequencies_specific_day.index.name = "time_diffs"
                time_diff_frequencies_specific_day.name = dayofweek
                time_diff_frequencies_list.append(pd.DataFrame(time_diff_frequencies_specific_day))

        time_diff_frequencies = time_diff_frequencies_list[0].join(time_diff_frequencies_list[1:], how='outer').sort_index()
        time_diff_frequencies = time_diff_frequencies.rename(columns={0:'Monday', 1:'Tuesday', 2:'Wednesday', 3:'Thursday', 4:'Friday', 5:'Saturday', 6:'Sunday'})
        time_diff_frequencies.columns.name = loc_id

        time_differences[loc_id] = time_diff_frequencies

    %store time_differences
    %store time_differences_details

In [ ]:
def plot_time_spacing(loc_id, colorbar_max):

    fig, ax = plt.subplots(figsize=(10, 5))

    # Assuming time_differences[loc_id] is a valid 2D array and using pandas DataFrame
    dat = time_differences[loc_id].T.iloc[:,1:]  # Converting to DataFrame if not already one

    full_columns = np.arange(24)

    dat = dat.reindex(columns=full_columns)

    norm = plt.Normalize(vmin=0, vmax=colorbar_max)

    cax = ax.imshow(dat, cmap='viridis', norm=norm)  # Display the data as an image

    # Adding a color bar
    fig.colorbar(cax, ax=ax)

    # Loop over data dimensions and create text annotations for each cell
    num_rows, num_cols = dat.shape
    for i in range(num_rows):
        for j in range(num_cols):
            # Only annotate if the value is not NaN
            if dat.iloc[i, j] == dat.iloc[i, j]:
                # Rounded value if not NaN
                rounded_value = round(100 * dat.iloc[i, j]) // 100
                ax.text(j, i, str(rounded_value), ha='center', va='center', color='w', fontsize=8)

    # Assigning labels for each column with the days of the week
    days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    # Setting x-axis ticks to be centered on each column
    ax.set_yticks(np.arange(num_rows))

    ax.set_yticklabels(days_of_week)

    # # Ensuring the labels are displayed at the top
    # ax.xaxis.set_ticks_position('top')

    ax.set_xticks(np.arange(1, 24))
    ax.set_xticklabels(np.arange(1, 24))
    ax.set_xlim(.5,23.5)

    ax.set_title('Number of Gaps in the Data of a Given Duration')
    ax.set_xlabel('Gap Duration in Hours')
    ax.set_ylabel('Day of the Week')

    plt.show()

In [ ]:
pd.options.display.float_format = '{:,.10f}'.format

# Turn off auto display
plt.ioff()

freq1 = 'D'
freq2 = 'W'
max_ylim1 = 0
max_ylim2 = 0
max_ylim3 = 0

for i, loc_id in enumerate(restaurants_by_4m_coverage):

    # Relevant DF
    df = sales_and_menu_data[loc_id]

    # Subset
    promo_datetime = pd.to_datetime(before_after_details.loc[loc_id, 'cross_over_date']).tz_localize('UTC')
    two_months_before = promo_datetime - pd.DateOffset(months=2)
    two_months_after = promo_datetime + pd.DateOffset(months=2)
    period = df.loc[two_months_before:two_months_after]

    # Calculate max y limit for plotting
    all_items1 = period.resample(freq1)['item_quantity'].sum()
    max_ylim1 = max(max_ylim1, all_items1.max())

    # Calculate max y limit for plotting
    all_items2 = period.resample(freq2)['item_quantity'].sum()
    max_ylim2 = max(max_ylim2, all_items2.max())

    # Calculate max y limit for plotting
    all_items3 = df.resample(freq2)['item_quantity'].sum()
    max_ylim3 = max(max_ylim3, all_items3.max())

    gaps_binarized = pd.DataFrame([time_differences[loc_id].iloc[0,].fillna(0), time_differences[loc_id].iloc[1:,].fillna(0).sum(axis=0)], index=['Gaps Less than an Hour', 'Gaps More than an Hour']).astype(int)

    # Summary stats
    print('\n\n\n\n')
    display(Markdown(f'## {loc_id}'))
    print(f'\nPlant-Based Analog: {before_after_details.loc[loc_id, "first_plant_based_mention"]}')
    promo_item_containing = sales_and_menu_data[loc_id][sales_and_menu_data[loc_id]['item_name'].str.contains(before_after_details.loc[loc_id,'first_plant_based_mention'])]
    print(promo_item_containing['item_name'].unique().tolist())
    
    plt.plot(promo_item_containing.resample('W')['item_quantity'].sum())
    plt.axvline(x=promo_datetime, color='red', linestyle='--', label='Promo Date')
    plt.title("Plant-Based Analog")
    plt.xticks(rotation=70)

    print(f"\n\n{locations.query('location_id == @loc_id')[['cuisine', 'city', 'state', 'restaurant_type']].to_markdown()}")
    print(f'\n{data_coverage_before.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_all.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_after.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_b2m.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_4m.loc[[loc_id]].to_markdown()}')
    print(f'\n{data_coverage_a2m.loc[[loc_id]].to_markdown()}')
    print(f'\n{gaps_binarized.to_markdown()}')
    print(f'\n{sales_and_menu_data[loc_id]["item_name"].value_counts().sort_values(ascending=False).iloc[:40].to_frame().to_markdown()}')

    # Daily
    total_items = time_differences[loc_id].sum().sum()
    colorbar_max = total_items / 10**3
    plot_time_spacing(loc_id, colorbar_max)
    plot_time_series(loc_id, df, max_ylim=max_ylim1, freq=freq1)
    plot_time_series(loc_id, df, max_ylim=max_ylim2, freq=freq2)
    plot_time_series(loc_id, df, max_ylim=max_ylim3, freq=freq2, subset=False)

    # Show
    plt.show()

In [ ]:
# sales_and_menu_data['L69HYJ4Y3TR91'][sales_and_menu_data['L69HYJ4Y3TR91']['item_name'].str.contains('Impossible')]
# sales_and_menu_data['L69HYJ4Y3TR91'].nlargest(50, 'item_name')